# Prompt Caching for Open-Weight LLMs — SGLang RadixAttention Benchmark

This notebook measures the latency & compute benefit of **engine-level prompt caching**
(SGLang RadixAttention) on two open-weight models:

- **Small:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`
- **Larger:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`


## Prerequisites
- **A CUDA GPU host with Docker** — e.g., an EC2 `g5.xlarge` (A10G, 24 GB) with the Deep Learning
  AMI, or a **SageMaker classic Notebook Instance** (`ml.g5.xlarge`) where Docker is available.
  > Note: **SageMaker Studio does not allow local Docker by default**, so this notebook is meant for
  > a GPU host with Docker. For a Studio-native reproduction, use
  > `sagemaker_studio_prompt_caching.ipynb` (endpoint-based) instead.
- We run SGLang from the official **`lmsysorg/sglang:latest`** Docker image, which ships a
  self-consistent CUDA + PyTorch + prebuilt-kernel stack. This avoids the just-in-time kernel-compile
  failures ("CUDA compiler and CUDA toolkit headers are incompatible") seen when `pip install`-ing
  SGLang into a mismatched CUDA environment.
- Outbound internet to pull the image (~40 GB) and model weights from Hugging Face (set `HF_TOKEN` if needed).

> Runtime note: first run pulls the image (~40 GB) and downloads weights (~3 GB for 1.5B, ~15 GB for
> 7B). Budget 15–25 min for the first launch.


## 1. Environment check

In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print("WARNING: nvidia-smi failed — you likely need a GPU instance.", e)


## 2. Pull the SGLang Docker image and install client deps

In [ ]:
import subprocess, sys
# Client-side deps only (the engine runs in Docker).
%pip install -q requests pandas matplotlib
# Pull the official SGLang image (self-consistent CUDA/torch/kernels). ~40 GB; first pull is slow.
IMAGE = "lmsysorg/sglang:latest"
print("pulling", IMAGE, "...")
subprocess.run(["docker", "pull", IMAGE], check=True)
print("image ready")


## 3. Configuration

In [ ]:
import os

MODELS = {
    "small": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "larger": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
}

PORT = 30000
HOST = "127.0.0.1"
BASE_URL = f"http://{HOST}:{PORT}"

# Workload knobs
PREFIX_TOKENS_DEFAULT = 2048     # shared system/RAG prefix size for W1/W2
VARIABLE_TOKENS       = 48       # unique user-query tokens per request
MAX_NEW_TOKENS        = 32       # fixed decode length so TTFT isolates prefill
PREFIX_SWEEP          = [256, 512, 1024, 2048, 4096, 8192]
N_REPEAT              = 5        # measured repetitions per condition (after warmup)

# Cost model inputs (edit to your instance price)
GPU_PRICE_PER_HOUR    = 1.006    # e.g., ml.g5.xlarge on-demand USD/hr — CHANGE to your rate
MANAGED_API_COST_PER_REQ = None  # optional: set a managed $/req to compute break-even

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("config ready")


## 4. Server launch / shutdown helpers (Docker)

In [ ]:
import subprocess, time, requests, atexit

CONTAINER = "sgl_bench"
HF_CACHE = os.path.expanduser("~/.cache/huggingface")

def start_server(model_path, disable_cache=False, mem_fraction=0.85):
    '''Launch SGLang in a Docker container and wait until healthy.'''
    stop_server()
    inner = [
        "python3", "-m", "sglang.launch_server",
        "--model-path", model_path,
        "--host", "0.0.0.0", "--port", str(PORT),
        "--mem-fraction-static", str(mem_fraction),
    ]
    if disable_cache:
        inner.append("--disable-radix-cache")
    cmd = [
        "docker", "run", "-d", "--gpus", "all", "--network", "host",
        "--shm-size", "16g", "-v", f"{HF_CACHE}:/root/.cache/huggingface",
        "--name", CONTAINER, IMAGE,
    ] + inner
    print("launching:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    deadline = time.time() + 1800
    while time.time() < deadline:
        alive = subprocess.run(["docker", "inspect", "-f", "{{.State.Running}}", CONTAINER],
                               capture_output=True, text=True).stdout.strip()
        if alive == "false":
            log = subprocess.run(["docker", "logs", "--tail", "40", CONTAINER],
                                 capture_output=True, text=True)
            raise RuntimeError("container exited early:\n" + log.stdout + log.stderr)
        try:
            if requests.get(f"{BASE_URL}/health_generate", timeout=2).status_code == 200:
                print("server ready"); return
        except Exception:
            pass
        time.sleep(3)
    raise TimeoutError("server did not become ready in time")

def stop_server():
    subprocess.run(["docker", "rm", "-f", CONTAINER],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

atexit.register(stop_server)

def flush_cache():
    try:
        requests.post(f"{BASE_URL}/flush_cache", timeout=10)
    except Exception as e:
        print("flush_cache warning:", e)

atexit.register(stop_server)
print("helpers ready")


## 5. Prompt builder and benchmark client
TTFT is measured from the streaming response; `cached_tokens` from SGLang `meta_info` is the direct prefill-savings proxy (tokens reused from the KV cache instead of recomputed).

In [ ]:
import time, json, requests

# A deterministic, tokenizer-agnostic filler. We size prefixes by word count and
# report the model-reported prompt_tokens for exactness.
_FILLER = ("You are a meticulous enterprise assistant. Follow the policy below exactly. "
           "Cite sources. Be concise. ") * 400  # long pool we slice from

def build_prompt(prefix_words, variable_words=VARIABLE_TOKENS, unique_tag=""):
    prefix = " ".join(_FILLER.split()[:prefix_words])
    variable = ("Question " + unique_tag + " ") + " ".join(
        ["detail%d" % i for i in range(variable_words)])
    # Stable prefix FIRST (cacheable), variable content LAST.
    return prefix + "\n\nUser query: " + variable

def generate_stream(prompt, max_new_tokens=MAX_NEW_TOKENS):
    '''Send a streaming /generate request; return (ttft_s, total_s, meta_info).'''
    payload = {
        "text": prompt,
        "sampling_params": {"max_new_tokens": max_new_tokens, "temperature": 0.0},
        "stream": True,
    }
    t0 = time.perf_counter()
    ttft = None
    meta = {}
    with requests.post(f"{BASE_URL}/generate", json=payload, stream=True, timeout=300) as r:
        r.raise_for_status()
        for raw in r.iter_lines(decode_unicode=True):
            if not raw:
                continue
            line = raw[5:].strip() if raw.startswith("data:") else raw.strip()
            if line == "[DONE]":
                break
            try:
                obj = json.loads(line)
            except Exception:
                continue
            if ttft is None:
                ttft = time.perf_counter() - t0   # first streamed chunk
            if "meta_info" in obj and obj["meta_info"]:
                meta = obj["meta_info"]
    total = time.perf_counter() - t0
    return ttft, total, meta

def measure(prefix_words, label, n=N_REPEAT):
    '''Cold (flushed) vs warm (repeated identical prefix) measurement.'''
    tag = label
    prompt = build_prompt(prefix_words, unique_tag=tag)

    # COLD: flush cache so the prefix must be prefilled from scratch.
    flush_cache()
    cold = generate_stream(prompt)

    # WARM: repeat identical prompt -> prefix served from RadixAttention cache.
    warm_samples = [generate_stream(prompt) for _ in range(n)]
    warm_ttft = sorted(s[0] for s in warm_samples)[len(warm_samples)//2]  # median
    warm_meta = warm_samples[-1][2]

    return {
        "prefix_words": prefix_words,
        "prompt_tokens": cold[2].get("prompt_tokens"),
        "ttft_cold_ms": round(cold[0]*1000, 1) if cold[0] else None,
        "ttft_warm_ms": round(warm_ttft*1000, 1) if warm_ttft else None,
        "cached_tokens_cold": cold[2].get("cached_tokens", 0),
        "cached_tokens_warm": warm_meta.get("cached_tokens", 0),
        "completion_tokens": warm_meta.get("completion_tokens"),
    }
print("client ready")


## 6. Run the benchmark
For each model: launch server → warm up → measure W1 (default prefix) → prefix-length sweep. Results collected into a DataFrame.

In [ ]:
import pandas as pd

rows = []
for size, model_path in MODELS.items():
    print(f"\n===== {size}: {model_path} =====")
    start_server(model_path, disable_cache=False)
    try:
        # warmup
        generate_stream(build_prompt(256, unique_tag="warmup"))

        # W1: default prefix, cold vs warm
        r1 = measure(PREFIX_TOKENS_DEFAULT, label="W1")
        r1.update(model=model_path, size=size, workload="W1")
        rows.append(r1)
        print("W1:", r1)

        # Prefix-length sweep
        for pw in PREFIX_SWEEP:
            r = measure(pw, label=f"sweep{pw}")
            r.update(model=model_path, size=size, workload="sweep")
            rows.append(r)
            print(f"sweep {pw}:", r["ttft_cold_ms"], "->", r["ttft_warm_ms"], "ms")
    finally:
        stop_server()

df = pd.DataFrame(rows)
df["ttft_reduction_pct"] = (1 - df["ttft_warm_ms"] / df["ttft_cold_ms"]) * 100
df.to_csv(f"{RESULTS_DIR}/raw_results.csv", index=False)
df


## 7. Paper table — TTFT cold vs warm (W1)

In [ ]:
w1 = df[df.workload == "W1"][
    ["size", "model", "prompt_tokens", "ttft_cold_ms", "ttft_warm_ms",
     "ttft_reduction_pct", "cached_tokens_warm"]
].round(1)
w1.to_csv(f"{RESULTS_DIR}/table_ttft_w1.csv", index=False)
w1


## 8. Prefix-length sweep (Figure 1)

In [ ]:
import matplotlib.pyplot as plt

sweep = df[df.workload == "sweep"]
fig, ax = plt.subplots(figsize=(7,4))
for size in sweep["size"].unique():
    s = sweep[sweep["size"] == size].sort_values("prefix_words")
    ax.plot(s["prefix_words"], s["ttft_reduction_pct"], marker="o", label=size)
ax.set_xlabel("Shared prefix (words)")
ax.set_ylabel("TTFT reduction on cache hit (%)")
ax.set_title("Prompt caching benefit vs. prefix length (SGLang RadixAttention)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig1_prefix_sweep.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
w1p = df[df.workload == "W1"]
x = range(len(w1p)); w = 0.35
ax.bar([i - w/2 for i in x], w1p["ttft_cold_ms"], w, label="cold (miss)")
ax.bar([i + w/2 for i in x], w1p["ttft_warm_ms"], w, label="warm (hit)")
ax.set_xticks(list(x)); ax.set_xticklabels(w1p["size"])
ax.set_ylabel("TTFT (ms)"); ax.set_title("TTFT: cold vs warm cache (W1)")
ax.legend(); plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_ttft_bar.png", dpi=150)
plt.show()


## 9. Cost model 
Approximate GPU-seconds/request from wall time, then translate TTFT savings into $/1M requests across cache-hit rates. Edit `GPU_PRICE_PER_HOUR` above for your instance.

In [ ]:
import numpy as np

def cost_curve(row, price_per_hour=GPU_PRICE_PER_HOUR):
    # Proxy GPU-seconds by TTFT (prefill-dominated) + fixed decode; use ms->s.
    gpu_sec_cold = (row["ttft_cold_ms"] or 0) / 1000.0
    gpu_sec_warm = (row["ttft_warm_ms"] or 0) / 1000.0
    hit = np.linspace(0, 1, 21)
    per_req = price_per_hour/3600.0 * (gpu_sec_warm*hit + gpu_sec_cold*(1-hit))
    per_1m = per_req * 1e6
    return hit, per_1m, gpu_sec_cold, gpu_sec_warm

fig, ax = plt.subplots(figsize=(7,4))
cost_rows = []
for _, row in df[df.workload=="W1"].iterrows():
    hit, per_1m, gsc, gsw = cost_curve(row)
    ax.plot(hit*100, per_1m, marker="o", label=row["size"])
    cost_rows.append({"size": row["size"], "gpu_sec_cold": round(gsc,4),
                      "gpu_sec_warm": round(gsw,4),
                      "cost_per_1M_at_80pct_hit": round(float(per_1m[16]),2)})
ax.set_xlabel("Cache-hit rate (%)"); ax.set_ylabel("Cost per 1M requests (USD)")
ax.set_title("Cost vs. cache-hit rate"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig2_cost.png", dpi=150); plt.show()

cost_df = pd.DataFrame(cost_rows)
cost_df.to_csv(f"{RESULTS_DIR}/table_cost.csv", index=False)
cost_df


## 10. Notes

**Interpretation checklist**
- H1: TTFT reduction should rise with prefix length (Fig 1).
- H2: absolute ms saved should be larger for the 7B model.
- H3: cost/1M falls as hit-rate rises  

**Optional control run:** re-run Section 6 with `start_server(..., disable_cache=True)` to confirm
warm≈cold when RadixAttention is off (isolates the cache as the cause of the speedup).
